# 08.10 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/10-sequences-rnn-lstm/exercises.ipynb)

In [ ]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

def rnn(xs, w_x=0.5, w_h=0.8, b=0.0):
    """A one-number RNN: h_t = tanh(w_x * x_t + w_h * h_(t-1) + b), starting from h_0 = 0."""
    h, states = 0.0, []
    for x in xs:
        h = np.tanh(w_x * x + w_h * h + b)
        states.append(round(float(h), 4))
    return states

def make_order_task(n, length=20, seed=0):
    """Sequences of random tokens 3-9 with one '1' and one '2'. Label 1 if '1' comes first."""
    rng = np.random.default_rng(seed)
    X = rng.integers(3, 10, (n, length))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        p, q = rng.choice(length, 2, replace=False)
        X[i, p], X[i, q] = 1, 2
        y[i] = int(p < q)
    return X, y

def bag_model(length, vocab=10):
    """Averages the token embeddings: it cannot see order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             layers.GlobalAveragePooling1D(),
                             layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])

def seq_model(length, cell=layers.LSTM, vocab=10):
    """Reads the tokens in order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             cell(16), layers.Dense(1, activation="sigmoid")])

def fit_score(model, X, y, Xt, yt, epochs=15):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X, y, epochs=epochs, batch_size=64, verbose=0)
    return model.evaluate(Xt, yt, verbose=0)[1]

## Exercise 1: how long is the memory?

By hand, then with `rnn`: feed `[1, 0, 0, 0, 0]` with $w_h = 0.2$ and with $w_h = 0.95$ (keep $w_x = 0.5$). After five steps, how much of the first input is left in each case?

In [ ]:
# your code here

## Exercise 2: GRU

Keras has a third cell, `layers.GRU`, a simpler LSTM with two gates. Train `seq_model(20, cell=layers.GRU)` on the order task. How does it compare with the LSTM?

In [ ]:
# your code here

## Exercise 3: longer sequences

Make the order task 100 tokens long. Compare `SimpleRNN` and `LSTM` (use `seq_model(100, cell=...)`, 15 epochs). Which one suffers from vanishing memory?

In [ ]:
# your code here

## Exercise 4: shorter reviews

Load IMDB with `maxlen=50` instead of 200 (only the last 50 words of each review) and train the LSTM again. What happens to accuracy, and why?

In [ ]:
# your code here